The electric field in a capacitor inspired by Joachim Schöberl

In [1]:
from netgen.meshing import Mesh as NGMesh, MeshPoint, Element1D, Element0D, Pnt
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw, FieldLines, AddFieldLines

import matplotlib.pylab as plt

In [2]:
def CapacitorGeometry():

    air = MoveTo(0, 0).RectangleC(30, 30).Face()
    air.edges.name = "Outer"
    air.faces.name = "air"

    el_u = MoveTo(0, 1).RectangleC(5, 0.5).Face()
    el_u.edges.name = "el_u"
    el_u.faces.name = "el_u"

    el_d = MoveTo(0, -1).RectangleC(5, 0.5).Face()
    el_d.edges.name = "el_d"
    el_d.faces.name = "el_d"

    dielectric = MoveTo(0, 0).RectangleC(4, 1.5).Face()
    dielectric.faces.name = "dielectric"

    shape = Glue([air - dielectric, dielectric])
    shape = shape - el_u - el_d

    shape.edges["el.*"].maxh=0.2
    shape.vertices["el.*"].maxh=0.2
    
    return shape


def CapacitorMesh(shape, h_max):
    
    mesh = Mesh(OCCGeometry(shape, dim=2).GenerateMesh(maxh=h_max))

    return mesh


def CapacitorSolver(mesh, FE_order, eps0, epsr):

    fes = H1(mesh, order=FE_order, dirichlet="el.*")

    u = fes.TrialFunction()
    v = fes.TestFunction()

    gfu = GridFunction(fes)
    gfu.Interpolate(mesh.BoundaryCF({"el_u":1, "el_d":-1 }), mesh.Boundaries(".*"))

    a = BilinearForm(eps0*epsr*grad(u)*grad(v)*dx).Assemble()
    
    inv = a.mat.Inverse(freedofs=fes.FreeDofs())
    gfu.vec.data -= inv@a.mat * gfu.vec

    return gfu

In [3]:
geo = CapacitorGeometry()
Draw(geo);

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': …

In [ ]:
h_max = 1
mesh = CapacitorMesh(geo, h_max)
Draw (mesh);

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [5]:
epsr_air = 1.0
epsr_dielectric = 2.0

epsr = mesh.MaterialCF({"air": epsr_air, "dielectric": epsr_dielectric})

Draw(epsr, mesh);

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [6]:
FE_order = 3
eps0 = 8.854e-12

gf_phi = CapacitorSolver(mesh, FE_order, eps0, epsr)

In [7]:
Draw (gf_phi, deformation=True, scale=5);

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [8]:
fes_flux = HDiv(mesh, order=FE_order-1)

gf_E = GridFunction(fes_flux)
gf_D = GridFunction(fes_flux)
gf_E.Set(-grad(gf_phi))
gf_D.Set(eps0*epsr*gf_E)

In [9]:
Draw (gf_E, mesh, vectors= { "grid_size" : 100});

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [10]:
Draw (Norm(gf_E), mesh, deformation=True, min=0, max=2);

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [11]:
Draw (gf_D, mesh, vectors= { "grid_size" : 100});

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [12]:
Draw (Norm(gf_D), mesh, vectors= { "grid_size" : 100});

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [13]:
N = 400
p = [(-10 + 0.05*i, -10 + 0.1*j, 0) for i in range(N) for j in range(N) ] 

fieldlines = gf_E._BuildFieldLines(mesh, p, num_fieldlines=500, length=0.3)

Draw(gf_E, mesh,  "X", draw_vol=True, draw_surf=True, objects=[fieldlines], \
     autoscale=True, min = 0, max = 2, settings={"Objects": {"Surface": False}});

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Surface': False…

In [14]:
energy = 0.5 * Integrate(eps0*epsr*InnerProduct(gf_E, gf_E), mesh)
print(energy)

1.2975953396565867e-10
